<a href="https://colab.research.google.com/github/thesubconsciousmind/agentic-project-debugger-collab/blob/main/agenticadassistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

----

## **The ADCP Media Buying Agent**

This section will build out a new agent focused on planning and executing media buys using the AdContextProtocol (ADCP).

### Implementation Plan:

1.  **Understand ADCP (from `https://docs.adcontextprotocol.org/docs/intro`):** We'll extract key entities and actions relevant to media buying (e.g., campaigns, ad slots, bidding, budgets, analytics).
2.  **Define ADCP Tools:** Create Python functions that simulate interactions with the ADCP, such as `create_campaign`, `adjust_budget`, `get_ad_slot_info`, `bid_on_ad_slot`, and `get_campaign_performance`.
3.  **Define ADCP Tool Schemas:** Develop the OpenAPI schemas for these tools, similar to how they are defined for the IT agent.
4.  **Implement `run_ad_agent` Function:** Adapt the existing agent's `run_it_agent` loop to use the new ADCP tools and a system prompt tailored for media buying.
5.  **Simulate ADCP Data:** Since we won't be connecting to a live ADCP, we'll create mock data structures to simulate campaign states, ad slot availability, and performance metrics.
6.  **Create Test Scenarios:** Develop various user prompts to test the agent's ability to plan, execute, and monitor media buys.

In [13]:
import os
import json
from openai import OpenAI
from google.colab import userdata
import requests
import json

In [14]:
client = OpenAI(api_key=userdata.get('OPENAI_APIKEY'))

### Initialize Data Stores

In [36]:
# --- Simulated ADCP Data Stores ---
adcp_campaigns = {}
next_campaign_id = 1

adcp_ad_slots = {
    "slot_101": {"publisher_id": "pub_A", "size": "300x250", "cpm_base": 5.00, "available": True},
    "slot_102": {"publisher_id": "pub_B", "size": "728x90", "cpm_base": 7.50, "available": True},
    "slot_103": {"publisher_id": "pub_A", "size": "320x100", "cpm_base": 3.20, "available": False}
}

==========================================
## PART 1: DEFINE THE ADCP TOOLS (BUSINESS LOGIC)
==========================================

In [25]:
import random
import json

# --- ADCP Tool Placeholders ---
def create_campaign(name: str, budget: float, start_date: str, end_date: str) -> str:
    """Creates a new advertising campaign in the ADCP system."""
    global next_campaign_id
    campaign_id = f"CMP-{next_campaign_id:03d}"
    next_campaign_id += 1
    campaign = {
        "campaign_id": campaign_id,
        "name": name,
        "budget": budget,
        "remaining_budget": budget,
        "start_date": start_date,
        "end_date": end_date,
        "status": "active",
        "ad_slots_booked": []
    }
    adcp_campaigns[campaign_id] = campaign
    print(f"-> TOOL: Created campaign {campaign_id} - {name} with budget {budget}.")
    return json.dumps({"status": "success", "campaign": campaign})

def get_campaign_status(campaign_id: str) -> str:
    """Retrieves the current status and performance metrics for a specific campaign."""
    campaign = adcp_campaigns.get(campaign_id)
    if not campaign:
        return json.dumps({"status": "error", "message": "Campaign not found."})

    # Simulate some performance metrics
    impressions = random.randint(10000, 100000)
    clicks = random.randint(100, 1000)
    cost = round(campaign["budget"] - campaign["remaining_budget"] + random.uniform(0, campaign["remaining_budget"] * 0.1), 2)
    if cost > campaign["budget"]:
      cost = campaign["budget"]
    cpc = round(cost / clicks, 2) if clicks > 0 else 0
    ctr = round((clicks / impressions) * 100, 2) if impressions > 0 else 0

    status_report = {
        "campaign_id": campaign_id,
        "name": campaign["name"],
        "status": campaign["status"],
        "budget": campaign["budget"],
        "remaining_budget": round(campaign["budget"] - cost, 2),
        "impressions": impressions,
        "clicks": clicks,
        "cost_incurred": cost,
        "cpc": cpc,
        "ctr": ctr,
        "ad_slots_booked": campaign["ad_slots_booked"]
    }
    print(f"-> TOOL: Retrieved status for campaign {campaign_id}.")
    return json.dumps({"status": "success", "report": status_report})

def get_available_ad_slots(min_cpm: float = 0.0, max_cpm: float = 1000.0, size: str = None) -> str:
    """Lists available ad slots based on criteria like CPM range and size."""
    available_slots = []
    for slot_id, details in adcp_ad_slots.items():
        if details["available"] and min_cpm <= details["cpm_base"] <= max_cpm:
            if size is None or details["size"] == size:
                available_slots.append({"slot_id": slot_id, "publisher_id": details["publisher_id"], "size": details["size"], "cpm_base": details["cpm_base"]})
    print(f"-> TOOL: Found {len(available_slots)} available ad slots.")
    return json.dumps({"status": "success", "ad_slots": available_slots})

def book_ad_slot(campaign_id: str, slot_id: str, bid_cpm: float) -> str:
    """Books a specific ad slot for a campaign with a given bid CPM."""
    campaign = adcp_campaigns.get(campaign_id)
    slot = adcp_ad_slots.get(slot_id)

    if not campaign:
        return json.dumps({"status": "error", "message": "Campaign not found."})
    if not slot or not slot["available"]:
        return json.dumps({"status": "error", "message": "Ad slot not found or not available."})
    if bid_cpm < slot["cpm_base"]:
        return json.dumps({"status": "error", "message": f"Bid CPM {bid_cpm} is below base CPM {slot['cpm_base']}."})

    # Simulate booking
    slot["available"] = False
    campaign["ad_slots_booked"].append({"slot_id": slot_id, "bid_cpm": bid_cpm})
    # Deduct some budget (simplified)
    campaign["remaining_budget"] = round(campaign["remaining_budget"] - (bid_cpm / 1000 * 50), 2) # Assume 50 impressions for simplicity

    print(f"-> TOOL: Booked ad slot {slot_id} for campaign {campaign_id} with bid {bid_cpm}.")
    return json.dumps({"status": "success", "slot_booked": {"slot_id": slot_id, "campaign_id": campaign_id, "bid_cpm": bid_cpm}})

def adjust_campaign_budget(campaign_id: str, new_budget: float) -> str:
    """Adjusts the total budget for an existing campaign."""
    campaign = adcp_campaigns.get(campaign_id)
    if not campaign:
        return json.dumps({"status": "error", "message": "Campaign not found."})

    campaign["budget"] = new_budget
    campaign["remaining_budget"] = new_budget # Reset remaining budget for simplicity in this simulation
    print(f"-> TOOL: Adjusted budget for campaign {campaign_id} to {new_budget}.")
    return json.dumps({"status": "success", "campaign_id": campaign_id, "new_budget": new_budget})

def list_all_campaigns() -> str:
    """Lists all active advertising campaigns."""
    campaign_list = [{
        "campaign_id": cid,
        "name": camp["name"],
        "status": camp["status"],
        "budget": camp["budget"],
        "remaining_budget": camp["remaining_budget"]
    } for cid, camp in adcp_campaigns.items()]
    print(f"-> TOOL: Listing all campaigns. Found {len(campaign_list)} campaigns.")
    return json.dumps({"status": "success", "campaigns": campaign_list})

def create_ad_creative(creative_text: str, campaign_id: str) -> str:
    """Generates ad creative content based on input text and associates it with a campaign."""
    campaign = adcp_campaigns.get(campaign_id)
    if not campaign:
        return json.dumps({"status": "error", "message": "Campaign not found."})

    creative_id = f"CR-{len(campaign.get('creatives', [])) + 1:03d}"
    # Simulate creative generation and storage
    creative_content = {
        "creative_id": creative_id,
        "text": creative_text,
        "status": "pending_review",
        "generated_on": "2024-07-22" # Placeholder date
    }
    if "creatives" not in campaign:
        campaign["creatives"] = []
    campaign["creatives"].append(creative_content)
    print(f"-> TOOL: Created ad creative {creative_id} for campaign {campaign_id} with text: '{creative_text[:30]}...' ")
    return json.dumps({"status": "success", "creative": creative_content, "campaign_id": campaign_id})

In [22]:
# Map ADCP functions for the agent execution loop
AVAILABLE_FUNCTIONS_ADCP = {
    "create_campaign": create_campaign,
    "get_campaign_status": get_campaign_status,
    "get_available_ad_slots": get_available_ad_slots,
    "book_ad_slot": book_ad_slot,
    "adjust_campaign_budget": adjust_campaign_budget,
    "list_all_campaigns": list_all_campaigns,
    "create_ad_creative": create_ad_creative
}

==========================================
## PART 2: DEFINE THE ADCP AGENT SCHEMA
==========================================

### ADCP Tool Schemas

In [23]:
tools_schema_adcp = [
    {
        "type": "function",
        "function": {
            "name": "create_campaign",
            "description": "Creates a new advertising campaign with a specified name, budget, start date, and end date.",
            "parameters": {
                "type": "object",
                "properties": {
                    "name": {"type": "string", "description": "The name of the campaign."},
                    "budget": {"type": "number", "description": "The total budget for the campaign in USD."},
                    "start_date": {"type": "string", "description": "The start date of the campaign in YYYY-MM-DD format."},
                    "end_date": {"type": "string", "description": "The end date of the campaign in YYYY-MM-DD format."}
                },
                "required": ["name", "budget", "start_date", "end_date"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_campaign_status",
            "description": "Retrieves the current status, budget usage, and performance metrics (impressions, clicks, cost) for a specific advertising campaign.",
            "parameters": {
                "type": "object",
                "properties": {
                    "campaign_id": {"type": "string", "description": "The unique identifier of the campaign, e.g., 'CMP-001'."}
                },
                "required": ["campaign_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_available_ad_slots",
            "description": "Lists ad slots that are currently available for booking, optionally filtered by CPM range and ad size.",
            "parameters": {
                "type": "object",
                "properties": {
                    "min_cpm": {"type": "number", "description": "Minimum Cost Per Mille (CPM) to filter by."},
                    "max_cpm": {"type": "number", "description": "Maximum Cost Per Mille (CPM) to filter by."},
                    "size": {"type": "string", "description": "Specific ad slot size to filter by, e.g., '300x250'."}
                },
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "book_ad_slot",
            "description": "Books a specific ad slot for a given campaign with a specified bid CPM. The bid CPM must be higher than the base CPM of the slot.",
            "parameters": {
                "type": "object",
                "properties": {
                    "campaign_id": {"type": "string", "description": "The unique identifier of the campaign for which to book the slot."},
                    "slot_id": {"type": "string", "description": "The unique identifier of the ad slot to book, e.g., 'slot_101'."},
                    "bid_cpm": {"type": "number", "description": "The Cost Per Mille (CPM) bid for this ad slot."}
                },
                "required": ["campaign_id", "slot_id", "bid_cpm"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "adjust_campaign_budget",
            "description": "Adjusts the total budget for an existing advertising campaign.",
            "parameters": {
                "type": "object",
                "properties": {
                    "campaign_id": {"type": "string", "description": "The unique identifier of the campaign."},
                    "new_budget": {"type": "number", "description": "The new total budget for the campaign in USD."}
                },
                "required": ["campaign_id", "new_budget"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "list_all_campaigns",
            "description": "Lists all active advertising campaigns with their current status and budget information.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "create_ad_creative",
            "description": "Generates ad creative content (e.g., text for an ad) based on input text and associates it with a campaign.",
            "parameters": {
                "type": "object",
                "properties": {
                    "creative_text": {"type": "string", "description": "The input text to base the creative content on."},
                    "campaign_id": {"type": "string", "description": "The ID of the campaign to associate this creative with, e.g., 'CMP-001'."}
                },
                "required": ["creative_text", "campaign_id"]
            }
        }
    }
]

==========================================
## PART 3: THE ADCP AGENT EXECUTION LOOP
==========================================

In [37]:
from openai import OpenAI

def run_ad_agent(user_query: str, client: OpenAI):
    print(f"\n--- New Ad Agent Request: {user_query} ---")

    messages = [
        {"role": "system", "content": """
You are an AI assistant specialized in AdContextProtocol (ADCP) media buying. Your task is to assist users in planning, executing, and monitoring advertising campaigns. You have access to tools for creating campaigns, managing budgets, booking ad slots, and checking campaign performance.

Operating rules:
1. If asked to create a campaign, use the `create_campaign` tool with appropriate parameters.
2. If asked to find ad slots, use `get_available_ad_slots`.
3. If asked to book an ad slot, use `book_ad_slot`.
4. If asked about campaign performance or status, use `get_campaign_status`.
5. If asked to change a campaign's budget, use `adjust_campaign_budget`.
6. If asked to list campaigns, use `list_all_campaigns`.
7. Always confirm actions taken and provide relevant details from the tool outputs.
8. If a required parameter for a tool is missing, ask the user for that information.
9. Provide a summary of actions taken and results after tool execution.
"""},
        {"role": "user", "content": user_query}
    ]

    while True:
        print("[AD Agent Thinking...]")
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=tools_schema_adcp,
            tool_choice="auto"
        )

        response_msg = response.choices[0].message
        messages.append(response_msg)

        if response_msg.tool_calls:
            for tool_call in response_msg.tool_calls:
                func_name = tool_call.function.name
                func_args = json.loads(tool_call.function.arguments)

                function_to_call = AVAILABLE_FUNCTIONS_ADCP.get(func_name)

                if function_to_call:
                    tool_output = function_to_call(**func_args)

                    messages.append({
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "name": func_name,
                        "content": tool_output
                    })

        else:
            print(f"\n[AD AGENT FINAL RESPONSE]: {response_msg.content}")
            break

==========================================
## PART 4: ADCP AGENT TEST SCENARIOS
==========================================

In [34]:
# Scenario 1: Create a new campaign
run_ad_agent("Create a new campaign named 'Summer Sale' with a budget of 5000 USD, running from 2024-07-01 to 2024-07-31.", client)


--- New Ad Agent Request: Create a new campaign named 'Summer Sale' with a budget of 5000 USD, running from 2024-07-01 to 2024-07-31. ---
[AD Agent Thinking...]
-> TOOL: Created campaign CMP-001 - Summer Sale with budget 5000.
[AD Agent Thinking...]

[AD AGENT FINAL RESPONSE]: The campaign 'Summer Sale' has been successfully created with the following details:

- **Campaign ID:** CMP-001
- **Budget:** 5000 USD
- **Remaining Budget:** 5000 USD
- **Start Date:** 2024-07-01
- **End Date:** 2024-07-31
- **Status:** Active
- **Ad Slots Booked:** None

If you need further assistance, such as booking ad slots or creating ad creatives, please let me know!


In [17]:
# Scenario 2: List all campaigns
run_ad_agent("Show me all my campaigns.", client)


--- New Ad Agent Request: Show me all my campaigns. ---

[AD Agent Thinking...]
-> TOOL: Listing all campaigns. Found 1 campaigns.

[AD Agent Thinking...]

[AD AGENT FINAL RESPONSE]: Here are all your campaigns:

1. **Campaign ID**: CMP-001
   - **Name**: Summer Sale
   - **Status**: Active
   - **Total Budget**: $5000
   - **Remaining Budget**: $5000

If you need further details or actions regarding this campaign, feel free to ask!


In [35]:
# Scenario 3: Create a new ad creative for an existing campaign
run_ad_agent("Generate an ad creative for the 'Summer Sale' campaign with the text 'Hot deals for summer! Shop now and save up to 50% on all items.'", client)


--- New Ad Agent Request: Generate an ad creative for the 'Summer Sale' campaign with the text 'Hot deals for summer! Shop now and save up to 50% on all items.' ---
[AD Agent Thinking...]

[AD AGENT FINAL RESPONSE]: To create the ad creative, I need the unique identifier (campaign ID) of the 'Summer Sale' campaign. Could you please provide that information?
